# Decile portfolios from CNN predictions

In this section, I construct 10 decile portfolios from the model's predicted probability of an up move (`pred_prob_up`).

The logic is:

1. Use `end_date` as the portfolio formation date.
2. Compute the realized 5-day forward return from `close_now` and `close_future`.
3. On each rebalance date, sort all stocks into 10 deciles based on `pred_prob_up`.
4. Compute the equal-weight return of each decile.
5. Annualize the mean return and Sharpe ratio.

This follows the general setup in Jiang (2023) and my supervisor's paper, where stocks are sorted into decile portfolios using model predictions, the portfolios are equal-weighted, and held for five trading days.

In [2]:
import pandas as pd
import numpy as np

## 1. Load the CSV and check the required columns

The file contains one row per stock-image observation.  
The most important columns here are:

- `end_date`: the date on which the image ends and the portfolio is formed
- `pred_prob_up`: the model's predicted probability that the future return is positive
- `close_now`: the close price at portfolio formation
- `close_future`: the close price 5 trading days later

I first load the data and make sure the required columns are present.

In [3]:
# Change this to your own file name
csv_path = r"res_96_color_candlestick_vol_1_ma_1_bb_0_rsi_0\test_predictions.csv"

df = pd.read_csv(csv_path)

required_cols = [
    "ticker", "end_date", "close_now", "close_future", "pred_prob_up"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Parse date
df["end_date"] = pd.to_datetime(df["end_date"])

# Keep only the columns we need here
df = df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]].copy()

print(df.head())
print("\nNumber of rows:", len(df))
print("Number of unique end_date values:", df["end_date"].nunique())
print("Number of unique tickers:", df["ticker"].nunique())

  ticker   end_date  close_now  close_future  pred_prob_up
0   ADSK 2023-10-24     205.04        197.63      0.468791
1    VOO 2020-04-13     253.19        258.86      0.611404
2    VEU 2022-11-01      46.37         47.96      0.676422
3    PPL 2024-05-08      28.52         29.57      0.496867
4    MDY 2024-02-05     499.26        517.76      0.452780

Number of rows: 14701
Number of unique end_date values: 442
Number of unique tickers: 400


## 2. Compute the realized 5-day forward return

For each stock observation, I compute the realized holding-period return as

$$
R_{i,t \to t+5} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

This is the return that each decile portfolio earns over the next 5 trading days.

In [4]:
# Basic cleaning
df = df.dropna(subset=["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]).copy()
df = df[(df["close_now"] > 0) & (df["close_future"] > 0)].copy()

# Remove duplicate stock-date observations if any
df = df.sort_values(["end_date", "ticker"]).drop_duplicates(subset=["ticker", "end_date"])

# Realized 5-day forward return
df["forward_return_5d"] = df["close_future"] / df["close_now"] - 1

print(df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up", "forward_return_5d"]].head())

      ticker   end_date  close_now  close_future  pred_prob_up  \
13584      A 2020-01-30      84.38         84.82      0.592107   
8674    ADSK 2020-01-30     198.99        206.00      0.602067   
7451     AEM 2020-01-30      61.17         59.60      0.500075   
3902    AMAT 2020-01-30      60.25         63.19      0.639207   
3041    ANET 2020-01-30     231.74        232.52      0.577938   

       forward_return_5d  
13584           0.005215  
8674            0.035228  
7451           -0.025666  
3902            0.048797  
3041            0.003366  


## 3A. Choose rebalance dates when the raw windows overlap

In this case, the dataset contains observations on many consecutive trading days, so the
underlying 5-day windows overlap.

However, if I want portfolio returns that do **not** overlap across holding periods, I should
rebalance every 5th trading date. This gives a clean sequence of non-overlapping 5-day
portfolio returns.

In [5]:
# Sort unique formation dates
unique_dates = np.sort(df["end_date"].unique())

# Keep every 5th trading date so that portfolio holding periods do not overlap
rebalance_dates = unique_dates[::5]

weekly_df = df[df["end_date"].isin(rebalance_dates)].copy()

print("Total unique dates in full sample:", len(unique_dates))
print("Rebalance dates used:", len(rebalance_dates))
print("Rows after keeping every 5th date:", len(weekly_df))
print(weekly_df.head())

Total unique dates in full sample: 442
Rebalance dates used: 89
Rows after keeping every 5th date: 3291
      ticker   end_date  close_now  close_future  pred_prob_up  \
13584      A 2020-01-30      84.38         84.82      0.592107   
8674    ADSK 2020-01-30     198.99        206.00      0.602067   
7451     AEM 2020-01-30      61.17         59.60      0.500075   
3902    AMAT 2020-01-30      60.25         63.19      0.639207   
3041    ANET 2020-01-30     231.74        232.52      0.577938   

       forward_return_5d  
13584           0.005215  
8674            0.035228  
7451           -0.025666  
3902            0.048797  
3041            0.003366  


## 3B. Choose rebalance dates when the data already has no 5-day overlap

In this case, the dataset is already constructed so that observations do not overlap across
the 5-day horizon.

Therefore, I can use **all available `end_date` values** as rebalance dates. No additional
subsampling is needed.

In [6]:
# Sort unique formation dates
unique_dates = np.sort(df["end_date"].unique())

# Use all dates, because the dataset is already non-overlapping
rebalance_dates = unique_dates

weekly_df = df[df["end_date"].isin(rebalance_dates)].copy()

print("Total unique dates in full sample:", len(unique_dates))
print("Rebalance dates used:", len(rebalance_dates))
print("Rows kept:", len(weekly_df))
print(weekly_df.head())

Total unique dates in full sample: 442
Rebalance dates used: 442
Rows kept: 14701
      ticker   end_date  close_now  close_future  pred_prob_up  \
13584      A 2020-01-30      84.38         84.82      0.592107   
8674    ADSK 2020-01-30     198.99        206.00      0.602067   
7451     AEM 2020-01-30      61.17         59.60      0.500075   
3902    AMAT 2020-01-30      60.25         63.19      0.639207   
3041    ANET 2020-01-30     231.74        232.52      0.577938   

       forward_return_5d  
13584           0.005215  
8674            0.035228  
7451           -0.025666  
3902            0.048797  
3041            0.003366  


In [7]:
print("COLUMNS:")
print(weekly_df.columns.tolist())

print("\nINDEX NAMES:")
print(weekly_df.index.names)

COLUMNS:
['ticker', 'end_date', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d']

INDEX NAMES:
[None]


In [8]:
print("Type:", type(weekly_df))
print("\nColumns as repr:")
print([repr(c) for c in weekly_df.columns])

print("\nIndex names:")
print(weekly_df.index.names)

print("\nDoes exact 'end_date' exist in columns?")
print("end_date" in weekly_df.columns)

Type: <class 'pandas.DataFrame'>

Columns as repr:
["'ticker'", "'end_date'", "'close_now'", "'close_future'", "'pred_prob_up'", "'forward_return_5d'"]

Index names:
[None]

Does exact 'end_date' exist in columns?
True


In [9]:
counts_per_date = weekly_df.groupby("end_date").size()
print(counts_per_date)


end_date
2020-01-30    60
2020-02-06    68
2020-02-13    57
2020-02-21    57
2020-02-28    61
              ..
2024-12-09     1
2024-12-11    50
2024-12-12     1
2024-12-18    61
2024-12-20     1
Length: 442, dtype: int64


## 4. Assign stocks to 10 deciles on each rebalance date

On each `end_date`, I sort stocks by `pred_prob_up`:

- Decile 1 = lowest predicted probability of going up
- Decile 10 = highest predicted probability of going up

I use `pd.qcut()` to split the cross-section into 10 approximately equal-sized groups.

A small practical issue is that some stocks can have identical prediction values.  
To make `qcut()` stable, I first rank the predictions using `rank(method="first")`.

In [10]:
# Make sure end_date is a normal column, not an index
if "end_date" not in weekly_df.columns:
    weekly_df = weekly_df.reset_index()

# Make a safe copy of end_date to use for grouping
weekly_df["end_date_copy"] = weekly_df["end_date"]

# Keep only dates with at least 10 stocks, otherwise 10 deciles are impossible
counts_per_date = weekly_df.groupby("end_date_copy").size()
valid_dates = counts_per_date[counts_per_date >= 10].index

weekly_df = weekly_df[weekly_df["end_date_copy"].isin(valid_dates)].copy()

def assign_deciles_one_date(group):
    group = group.copy()

    # Rank first to avoid problems when many predictions are tied
    ranked = group["pred_prob_up"].rank(method="first")

    # Decile 1 = lowest prediction, Decile 10 = highest prediction
    group["decile"] = pd.qcut(ranked, q=10, labels=False) + 1
    return group

weekly_df = (
    weekly_df
    .groupby("end_date_copy", group_keys=False)
    .apply(assign_deciles_one_date)
    .reset_index(drop=True)
)

print(weekly_df.head(10))
print("\nNumber of rows:", len(weekly_df))

  ticker   end_date  close_now  close_future  pred_prob_up  forward_return_5d  \
0      A 2020-01-30      84.38         84.82      0.592107           0.005215   
1   ADSK 2020-01-30     198.99        206.00      0.602067           0.035228   
2    AEM 2020-01-30      61.17         59.60      0.500075          -0.025666   
3   AMAT 2020-01-30      60.25         63.19      0.639207           0.048797   
4   ANET 2020-01-30     231.74        232.52      0.577938           0.003366   
5   ANSS 2020-01-30     280.14        285.88      0.614909           0.020490   
6    AON 2020-01-30     219.63        229.60      0.487886           0.045395   
7    APH 2020-01-30     101.87        103.96      0.561540           0.020516   
8   ASML 2020-01-30     294.38        309.55      0.619825           0.051532   
9    BNS 2020-01-30      55.11         55.66      0.600527           0.009980   

   decile  
0       5  
1       7  
2       2  
3       9  
4       5  
5       7  
6       2  
7       4  


In [11]:
print("COLUMNS:")
print(list(weekly_df.columns))

print("\nINDEX NAME(S):")
print(weekly_df.index.names)

print("\nHEAD:")
print(weekly_df.head())

COLUMNS:
['ticker', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d', 'decile']

INDEX NAME(S):
[None]

HEAD:
  ticker  close_now  close_future  pred_prob_up  forward_return_5d  decile
0      A      84.38         84.82      0.592107           0.005215       5
1   ADSK     198.99        206.00      0.602067           0.035228       7
2    AEM      61.17         59.60      0.500075          -0.025666       2
3   AMAT      60.25         63.19      0.639207           0.048797       9
4   ANET     231.74        232.52      0.577938           0.003366       5


In [11]:
# Quick diagnostic: number of stocks in each decile on each date
decile_counts = weekly_df.groupby(["end_date", "decile"]).size().unstack()
print(decile_counts.head())

decile      1   2   3   4   5   6   7   8   9   10
end_date                                          
2020-01-30   6   6   6   6   6   6   6   6   6   6
2020-02-06   7   7   7   6   7   7   6   7   7   7
2020-02-13   6   6   5   6   6   5   6   5   6   6
2020-02-21   6   6   5   6   6   5   6   5   6   6
2020-02-28   7   6   6   6   6   6   6   6   6   6


## 5. Compute equal-weight decile returns on each rebalance date

For each formation date and decile, I take the simple average of the realized 5-day forward returns across all stocks in that decile.

This gives me one 5-day portfolio return for each decile on each rebalance date.

In [12]:
decile_returns_by_date = (
    weekly_df
    .groupby(["end_date", "decile"])["forward_return_5d"]
    .mean()
    .reset_index()
    .sort_values(["end_date", "decile"])
)

print(decile_returns_by_date.head(15))

     end_date  decile  forward_return_5d
0  2020-01-30       1           0.006863
1  2020-01-30       2           0.006460
2  2020-01-30       3           0.011495
3  2020-01-30       4           0.017658
4  2020-01-30       5           0.008828
5  2020-01-30       6           0.007641
6  2020-01-30       7           0.018158
7  2020-01-30       8           0.007026
8  2020-01-30       9           0.041920
9  2020-01-30      10           0.037794
10 2020-02-06       1           0.014209
11 2020-02-06       2           0.009377
12 2020-02-06       3           0.002911
13 2020-02-06       4           0.005948
14 2020-02-06       5           0.011453


## 6. Put the decile returns into wide format and construct High-minus-Low

Now I reshape the data so that each column is one decile:

- column 1 = Low
- column 10 = High

Then I create the spread portfolio:

$$H-L = \text{Decile 10} - \text{Decile 1}$$

This is the same long-short spread that is reported in the papers.

In [13]:
decile_matrix = decile_returns_by_date.pivot(
    index="end_date",
    columns="decile",
    values="forward_return_5d"
).sort_index()

# Add High-minus-Low spread
decile_matrix["H-L"] = decile_matrix[10] - decile_matrix[1]

# Optional: rename columns for nicer display
decile_matrix = decile_matrix.rename(columns={1: "Low", 10: "High"})

print(decile_matrix.head())

decile           Low         2         3         4         5         6  \
end_date                                                                 
2020-01-30  0.006863  0.006460  0.011495  0.017658  0.008828  0.007641   
2020-02-06  0.014209  0.009377  0.002911  0.005948  0.011453  0.005690   
2020-02-13 -0.000890 -0.007789 -0.016733 -0.000887  0.027454 -0.016429   
2020-02-21 -0.127288 -0.076384 -0.107598 -0.098345 -0.114895 -0.133748   
2020-02-28 -0.046207 -0.054182 -0.027140 -0.053753  0.005122  0.022481   

decile             7         8         9      High       H-L  
end_date                                                      
2020-01-30  0.018158  0.007026  0.041920  0.037794  0.030931  
2020-02-06  0.003075 -0.009598 -0.002007  0.004273 -0.009936  
2020-02-13 -0.003222 -0.001293  0.004054 -0.000766  0.000125  
2020-02-21 -0.090233 -0.120464 -0.144354 -0.125173  0.002114  
2020-02-28 -0.013649  0.014713  0.037824  0.043655  0.089862  


## 7. Compute annualized return and annualized Sharpe ratio

Each row in `decile_matrix` is a **5-trading-day portfolio return**.

So I annualize using:

$$
\text{periods per year} = \frac{252}{5}
$$

Then:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

This is the standard way to annualize fixed-horizon portfolio returns.

In [14]:
periods_per_year = 252 / 5  # 5-trading-day holding period

def annualized_stats(return_series):
    s = pd.Series(return_series).dropna()
    
    mean_5d = s.mean()
    std_5d = s.std(ddof=1)
    
    ann_return = mean_5d * periods_per_year
    
    if std_5d == 0 or np.isnan(std_5d):
        ann_sharpe = np.nan
    else:
        ann_sharpe = (mean_5d / std_5d) * np.sqrt(periods_per_year)
    
    return pd.Series({
        "Mean_5d_Return": mean_5d,
        "Std_5d_Return": std_5d,
        "Annualized_Return": ann_return,
        "Annualized_Sharpe": ann_sharpe,
        "N_periods": len(s)
    })

summary = decile_matrix.apply(annualized_stats, axis=0).T
summary

,Mean_5d_Return,Std_5d_Return,Annualized_Return,Annualized_Sharpe,N_periods
decile,,,,,
Low,0.000072,0.032119,0.003628,0.015910,247.0
2,0.002572,0.032904,0.129619,0.554881,247.0
3,0.025789,0.358450,1.299768,0.510766,247.0
4,0.004214,0.033602,0.212391,0.890336,247.0
5,0.003675,0.032081,0.185215,0.813239,247.0
6,0.002047,0.034492,0.103148,0.421240,247.0
7,0.001583,0.034571,0.079766,0.325005,247.0
8,0.004038,0.036232,0.203537,0.791290,247.0
9,0.004360,0.036513,0.219724,0.847640,247.0


## 8. Make the final table look like the paper

To make the output easier to compare with the paper tables, I keep only the annualized return and annualized Sharpe ratio, and I convert the return to percent.

In [15]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                    0.3628             0.0159      247.0
2                     12.9619             0.5549      247.0
3                    129.9768             0.5108      247.0
4                     21.2391             0.8903      247.0
5                     18.5215             0.8132      247.0
6                     10.3148             0.4212      247.0
7                      7.9766             0.3250      247.0
8                     20.3537             0.7913      247.0
9                     21.9724             0.8476      247.0
High                   9.7736             0.3695      247.0
H-L                    9.4108             0.4488      247.0


## 9. Interpretation of the output

The final table should be read as follows:

- `Low` is the decile with the lowest predicted probability of an up move.
- `High` is the decile with the highest predicted probability of an up move.
- `H-L` is a long-short strategy that buys the highest decile and shorts the lowest decile.
- `Annualized_Return_pct` is the annualized mean return in percent.
- `Annualized_Sharpe` is the annualized Sharpe ratio.

If the model is useful, I should generally see returns and Sharpe ratios improve as I move from `Low` to `High`.

In [16]:
# Average number of stocks in each decile
avg_names_per_decile = (
    weekly_df.groupby(["end_date", "decile"]).size()
    .groupby("decile")
    .mean()
)

print("Average number of stocks per decile:")
print(avg_names_per_decile.round(2))

# Check monotonicity visually
display_cols = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in decile_matrix.columns]
display(decile_matrix[display_cols].head())

Average number of stocks per decile:
decile
1     6.35
2     5.80
3     5.74
4     5.79
5     5.95
6     5.61
7     5.71
8     5.83
9     5.71
10    6.19
dtype: float64


decile,Low,2,3,4,5,6,7,8,9,High,H-L
end_date,,,,,,,,,,,
2020-01-30,0.006863,0.006460,0.011495,0.017658,0.008828,0.007641,0.018158,0.007026,0.041920,0.037794,0.030931
2020-02-06,0.014209,0.009377,0.002911,0.005948,0.011453,0.005690,0.003075,-0.009598,-0.002007,0.004273,-0.009936
2020-02-13,-0.000890,-0.007789,-0.016733,-0.000887,0.027454,-0.016429,-0.003222,-0.001293,0.004054,-0.000766,0.000125
2020-02-21,-0.127288,-0.076384,-0.107598,-0.098345,-0.114895,-0.133748,-0.090233,-0.120464,-0.144354,-0.125173,0.002114
2020-02-28,-0.046207,-0.054182,-0.027140,-0.053753,0.005122,0.022481,-0.013649,0.014713,0.037824,0.043655,0.089862


## Always-long equal-weight benchmark

As a reference benchmark, I also calculate the performance of an **always-long, equal-weight portfolio across all stocks**.

This benchmark does **not** sort stocks into deciles.  
Instead, on each portfolio formation date, it simply buys **all available stocks** and assigns each stock the same weight.

That is why this benchmark produces only **one portfolio return series**, and therefore only **one annualized return** and **one Sharpe ratio**.

### Step 1: Compute the 5-day forward return for each stock

For each stock $i$ on formation date $t$, the realized 5-day forward return is:

$$
r_{i,t} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

where:

- $\text{close\_now}_{i,t}$ is the closing price at the portfolio formation date
- $\text{close\_future}_{i,t}$ is the closing price 5 trading days later

### Step 2: Compute the equal-weight benchmark return on each date

On each formation date $t$, the always-long benchmark return is the simple average of all stock returns on that date:

$$
r^{EW}_t = \frac{1}{N_t} \sum_{i=1}^{N_t} r_{i,t}
$$

where $N_t$ is the number of available stocks on date $t$.

So instead of creating 10 decile portfolios, I create only **one** portfolio each period:

- long all stocks
- equal weight each stock
- hold for 5 trading days

This gives a time series of benchmark returns:

$$
r^{EW}_{t_1}, r^{EW}_{t_2}, r^{EW}_{t_3}, \dots
$$

### Step 3: Annualize the mean return

Because each portfolio is held for 5 trading days, the number of holding periods per year is approximately:

$$
\frac{252}{5}
$$

where 252 is the standard number of trading days in a year.

The annualized return is therefore:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

where $\bar{r}_{5d}$ is the average 5-day benchmark return across all periods.

### Step 4: Annualize the Sharpe ratio

The Sharpe ratio measures return relative to volatility.

Let $\sigma_{5d}$ denote the standard deviation of the 5-day benchmark returns.  
Then the annualized Sharpe ratio is:

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

### Why does this benchmark only give one return and one Sharpe ratio?

The decile analysis gives many values because stocks are split into many portfolios:

- Decile 1
- Decile 2
- ...
- Decile 10
- High-minus-Low

So each decile has its own return series and its own Sharpe ratio.

In contrast, the always-long benchmark does **not** split stocks into groups.  
It simply averages all stocks into one equal-weight portfolio on each date.

Therefore, it produces:

- one portfolio return series
- one annualized return
- one annualized Sharpe ratio

### Interpretation

This benchmark is useful because it shows how well a simple passive strategy performs without using the model.

If the model is useful in a long-only sense, then the **High decile** should ideally outperform this benchmark in terms of:

- annualized return
- Sharpe ratio

If the model is useful as a ranking model, then returns should generally improve from the **Low** decile to the **High** decile, and the **High-minus-Low** spread should be positive and economically meaningful.

In [17]:
import pandas as pd
import numpy as np

# df must contain: end_date, close_now, close_future
benchmark_df = df.copy()

benchmark_df["end_date"] = pd.to_datetime(benchmark_df["end_date"])
benchmark_df = benchmark_df.dropna(subset=["end_date", "close_now", "close_future"])
benchmark_df = benchmark_df[(benchmark_df["close_now"] > 0) & (benchmark_df["close_future"] > 0)].copy()

# 5-day forward stock return
benchmark_df["forward_return_5d"] = benchmark_df["close_future"] / benchmark_df["close_now"] - 1

# Always-long equal-weight portfolio across all stocks on each date
ew_returns = (
    benchmark_df
    .groupby("end_date")["forward_return_5d"]
    .mean()
    .sort_index()
)

# Annualization for 5-trading-day holding periods
periods_per_year = 252 / 5

annualized_return = ew_returns.mean() * periods_per_year
annualized_sharpe = (ew_returns.mean() / ew_returns.std(ddof=1)) * np.sqrt(periods_per_year)

print("Always-long equal-weight benchmark")
print(f"Annualized return: {annualized_return:.4f}  ({annualized_return*100:.2f}%)")
print(f"Annualized Sharpe: {annualized_sharpe:.4f}")
print(f"Number of periods: {len(ew_returns)}")

Always-long equal-weight benchmark
Annualized return: 0.3821  (38.21%)
Annualized Sharpe: 1.0836
Number of periods: 442
